# Etapa 5f — Ronda para los vasos finos (plan, desvío 14.8)

**Pregunta.** El algoritmo detecta un tercio de los vasos finos (34 %), y eso es lo que más baja la concordancia de las medidas clínicas (etapa 7). ¿Se puede mejorar sin perder en el resto?

**Candidatas** (los mismos 26 mapas, la misma validación cruzada por personas y la misma elección de la fracción que `05e`):

| Candidata | Resolución | Píxeles de vaso para entrenar |
|---|---|---|
| A. Actual | mitad | sorteados al azar (la mayoría son de vasos gruesos) |
| B. Equilibrado | mitad | un tercio de cada clase de grosor (finos, medianos, gruesos) |
| C. Completa | completa | sorteados al azar |
| D. Completa y equilibrado | completa | un tercio de cada clase |

El fondo para entrenar y la muestra para elegir el umbral son los mismos en las cuatro. A resolución completa los mapas se calculan de a uno (`feature_scores`, `sample_features`), para que entren en memoria.

**Cuándo se adopta** (fijado antes de correr, desvío 14.8). Una candidata reemplaza a A si mejora la sensibilidad de línea central en vasos finos con un intervalo al 95 % que no incluye el cero, y si Matthews y el Dice de línea central no son inferiores a los de A (límite inferior del intervalo de la diferencia por encima de −0,03). Si cumplen varias, gana la más barata (en este orden: B, C, D) entre las que no son inferiores a la mejor en vasos finos (mismo margen). Si ninguna cumple, queda A. La decisión se toma con la fracción fija; el valor fijo se informa. Después de esta ronda el algoritmo se cierra sin excepciones.


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_fraction, choose_value, fit_weights
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
CANDS = {"A. actual": (2, False), "B. equilibrado": (2, True),
         "C. completa": (1, False), "D. completa y equilibrado": (1, True)}
WORKERS = 2    # a resolución completa cada proceso usa cerca de 1 GB

## Pasada 1: muestras de entrenamiento y para el umbral, a las dos resoluciones

In [2]:
def pass1(image_id):
    from src.pipeline import sample_features, training_pixels
    out = {}
    for factor in (2, 1):
        v = sample_features(image_id, factor)
        nat = training_pixels(image_id, values=v, balanced=False)
        bal = training_pixels(image_id, values=v, balanced=True)
        out[factor] = {"natural": nat, "equilibrado": {**nat, "x": bal["x"], "y": bal["y"]}}
    return image_id, out

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
print(f"Pasada 1: {time.time() - t:.0f} s")
samples = {c: {i: P1[i][f]["equilibrado" if b else "natural"] for i in ids} for c, (f, b) in CANDS.items()}
del P1

Pasada 1: 2491 s


In [3]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    for c in CANDS:
        w = fit_weights(samples[c], tr)
        fit[(k, c)] = (w, choose_fraction(samples[c], w, tr), choose_value(samples[c], w, tr))
print("Fracción elegida en cada vuelta:")
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)])

Fracción elegida en cada vuelta:


,A. actual,B. equilibrado,C. completa,D. completa y equilibrado
vuelta 0,0.10,0.10,0.09,0.10
vuelta 1,0.10,0.10,0.09,0.10
vuelta 2,0.09,0.09,0.09,0.09
vuelta 3,0.09,0.09,0.09,0.10
vuelta 4,0.09,0.09,0.09,0.09


## Pasada 2: personas de afuera, a resolución original

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.segment import segmentar_fraccion
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    rows = []
    for factor in (2, 1):
        names = [c for c, (f, _) in CANDS.items() if f == factor]
        probs, _, _ = feature_scores(rgb, [fit[(k, c)][0] for c in names], factor)
        for c, p in zip(names, probs):
            p = p if p.shape == fov.shape else upscale(p, fov.shape)
            _, frac, value = fit[(k, c)]
            for rule, m in (("fracción", segmentar_fraccion(p, auto, frac)[0]), ("valor fijo", (p > value) & auto)):
                rows.append({"image": image_id, "candidata": c, "regla": rule, "matthews": matthews(m, truth, fov),
                             "dice_linea_central": centerline_dice(m, truth, fov), "finos": m[thin].mean()})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05f_vasos_finos.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 2775 s


## Resultados

In [5]:
ORDER = list(CANDS)
def compare(rule):
    out = []
    for meas in ("matthews", "dice_linea_central", "finos"):
        w = R[R["regla"] == rule].pivot(index="image", columns="candidata", values=meas)[ORDER]
        d = w.sub(w["A. actual"], axis=0)
        lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
        for i, c in enumerate(ORDER):
            out.append({"medida": meas, "candidata": c, "media": w[c].mean(), "vs A": d[c].mean(),
                        "inferior": lo[i], "superior": hi[i]})
    return pd.DataFrame(out).set_index(["medida", "candidata"])

table = compare("fracción")
print("Con fracción fija (la que decide):")
table.round(4)

Con fracción fija (la que decide):


media    vs A  inferior  superior
medida             candidata                                                    
matthews           A. actual                  0.7255  0.0000    0.0000    0.0000
                   B. equilibrado             0.7255 -0.0000   -0.0025    0.0018
                   C. completa                0.7256  0.0001   -0.0023    0.0028
                   D. completa y equilibrado  0.7250 -0.0005   -0.0045    0.0032
dice_linea_central A. actual                  0.7326  0.0000    0.0000    0.0000
                   B. equilibrado             0.7361  0.0035   -0.0005    0.0073
                   C. completa                0.7134 -0.0192   -0.0209   -0.0176
                   D. completa y equilibrado  0.7211 -0.0115   -0.0195   -0.0045
finos              A. actual                  0.3369  0.0000    0.0000    0.0000
                   B. equilibrado             0.4040  0.0671    0.0602    0.0745
                   C. completa                0.2859 -0.0510   -0.0662   -0.0373
                   D. completa y equilibrado  0.3914  0.0544    0.0405    0.0710

In [6]:
print("Con valor fijo (se informa):")
compare("valor fijo").round(4)

Con valor fijo (se informa):


media    vs A  inferior  superior
medida             candidata                                                    
matthews           A. actual                  0.7304  0.0000    0.0000    0.0000
                   B. equilibrado             0.7302 -0.0002   -0.0029    0.0019
                   C. completa                0.7290 -0.0014   -0.0032    0.0006
                   D. completa y equilibrado  0.7312  0.0008   -0.0041    0.0049
dice_linea_central A. actual                  0.7313  0.0000    0.0000    0.0000
                   B. equilibrado             0.7383  0.0070    0.0035    0.0100
                   C. completa                0.7179 -0.0134   -0.0163   -0.0111
                   D. completa y equilibrado  0.7312 -0.0001   -0.0078    0.0060
finos              A. actual                  0.2913  0.0000    0.0000    0.0000
                   B. equilibrado             0.3569  0.0657    0.0592    0.0723
                   C. completa                0.2873 -0.0040   -0.0096    0.0011
                   D. completa y equilibrado  0.3911  0.0999    0.0918    0.1075

In [7]:
ok = {}
for c in ORDER[1:]:
    ok[c] = (table.loc[("finos", c), "inferior"] > 0
             and table.loc[("matthews", c), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
             and table.loc[("dice_linea_central", c), "inferior"] > -C.NONINF_MARGIN_MATTHEWS)
print("Cumple la regla:", ok)
passing = [c for c in ORDER[1:] if ok[c]]
if passing:
    fin = R[R["regla"] == "fracción"].pivot(index="image", columns="candidata", values="finos")
    best = max(passing, key=lambda c: fin[c].mean())
    d = fin[passing].sub(fin[best], axis=0)
    lo, _ = percentile_interval(person_bootstrap_means(d, dev["person"]))
    noninf = [c for c, l in zip(passing, lo) if l > -C.NONINF_MARGIN_MATTHEWS]
    chosen = noninf[0]     # ORDER va de la más barata a la más cara
else:
    chosen = "A. actual"
print("Se adopta:", chosen)

Cumple la regla: {'B. equilibrado': np.True_, 'C. completa': np.False_, 'D. completa y equilibrado': np.True_}
Se adopta: B. equilibrado


In [8]:
print("Por grupo, con fracción fija (media):")
R[R["regla"] == "fracción"].join(dev["group"], on="image").pivot_table(
    index="candidata", columns="group", values=["matthews", "finos"]).loc[ORDER].round(3)

Por grupo, con fracción fija (media):


finos               matthews              
group                         dr      g      h       dr      g      h
candidata                                                            
A. actual                  0.352  0.368  0.291    0.678  0.728  0.771
B. equilibrado             0.405  0.433  0.374    0.673  0.732  0.771
C. completa                0.303  0.316  0.238    0.679  0.730  0.768
D. completa y equilibrado  0.391  0.426  0.358    0.670  0.730  0.775

## Decisión (con la regla fijada antes de correr)

**Verificación:** A da 0,7255 con fracción fija y 0,7304 con valor fijo, lo mismo que `05e` y `05d`: calcular los mapas de a uno no cambió nada.

| Candidata | Vasos finos contra A (intervalo) | Matthews contra A | Dice de línea central contra A | ¿Cumple? |
|---|---|---|---|---|
| B. Equilibrado | +0,067 (+0,060 a +0,075) | −0,000 (−0,003 a +0,002) | +0,004 (−0,001 a +0,007) | sí |
| C. Completa | −0,051 (−0,066 a −0,037) | +0,000 | −0,019 | no |
| D. Completa y equilibrado | +0,054 (+0,041 a +0,071) | −0,001 (−0,005 a +0,003) | −0,012 (−0,020 a −0,005) | sí |

Cumplen B y D. B es la mejor en vasos finos y la más barata: **se adopta B** (plan, desvío 14.8) y el algoritmo queda cerrado.

- **Con B:** vasos finos 40 % (contra 34 %), Matthews 0,7255 (igual) y Dice de línea central 0,736. Los finos mejoran en los tres grupos (retinopatía 35 % → 41 %, glaucoma 37 % → 43 %, sanos 29 % → 37 %). En retinopatía, Matthews baja apenas (0,678 → 0,673).
- **Con valor fijo** (se informa): B da finos +0,066 y Dice de línea central +0,007, con Matthews igual. D llega a +0,10 en finos con valor fijo, pero la regla decide con la fracción.
- **La resolución completa no ayudó, y sola empeora los finos.** Lo más probable es que, a tamaño completo, la escala más fina de los filtros responda sobre todo al ruido y a los bloques de la compresión .jpg, que a la mitad de resolución se suavizan al reducir. Va al informe como resultado negativo.
- **Por qué funciona el equilibrado:** al entrenar con un tercio de píxeles de cada grosor, la regresión aprende a pesar las escalas finas en lugar de optimizar sobre todo para los vasos gruesos, que son la mayoría de los píxeles. No cuesta nada en tiempo: el algoritmo es el mismo, solo cambian los pesos.

**Cuidado:** es otra elección hecha con las mismas 30 imágenes, así que los valores de desarrollo son algo optimistas; la medida que vale es la de prueba.